# EMiF Class 13 - Nonlinear econometrics in finance

This notebook accompanies the slide deck. The objective is not to build a production-grade trading signal. The objective is pedagogical: show how the same financial dataset can be used to move from a linear model to threshold, smooth, quantile and non-parametric views of the conditional distribution.

Dataset: Fama-French Research Data 5 Factors 2x3, daily frequency. Returns are provided in percent in the raw CSV and converted into decimals below.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg

DATA = 'data/F-F_Research_Data_5_Factors_2x3_daily.csv'
FIG = 'Figures'
os.makedirs(FIG, exist_ok=True)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Load Fama-French daily factors

The CSV contains header comments and a copyright footer, so we detect the data block explicitly. The factors are: market excess return, size, value, profitability and investment.

In [ ]:
with open(DATA, 'r', encoding='utf-8') as f:
    lines = f.readlines()

start = None
end = None
for i, line in enumerate(lines):
    if line.startswith(',Mkt-RF'):
        start = i
    if start is not None and line.startswith('Copyright'):
        end = i
        break

if end is None:
    end = len(lines)

ff = pd.read_csv(DATA, skiprows=start, nrows=end-start-2)
ff = ff.rename(columns={ff.columns[0]: 'Date'})
ff['Date'] = pd.to_datetime(ff['Date'].astype(str), format='%Y%m%d')
ff = ff.set_index('Date')
ff = ff.apply(pd.to_numeric, errors='coerce') / 100.0
ff = ff.dropna()

factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
print(ff.index.min(), ff.index.max(), ff.shape)
ff.head()

## 2. First diagnostic: unconditional distribution and volatility clustering

A linear Gaussian model is often a useful benchmark. In daily finance, it is rarely the whole story: tails are fat, skewness can be negative, and volatility is strongly time-varying.

In [ ]:
summary = pd.DataFrame({
    'ann_return_%': 100*((1+ff[factors]).prod()**(252/len(ff))-1),
    'ann_vol_%': 100*ff[factors].std()*np.sqrt(252),
    'skewness': ff[factors].skew(),
    'kurtosis': ff[factors].kurtosis()+3
})
summary.round(2)

In [ ]:
cum = (1 + ff[factors]).cumprod().resample('W-FRI').last()
ax = cum.plot(logy=True, figsize=(9, 5))
ax.set_title('Fama-French daily factors: cumulative factor premiums')
ax.set_ylabel('Cumulative value, log scale')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f'{FIG}/ff_cumulative_factors.pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
roll_vol = (ff['Mkt-RF'].rolling(252).std() * np.sqrt(252)).resample('W-FRI').last()
ax = (100*roll_vol).plot(figsize=(9, 5))
ax.set_title('Nonlinearity starts with conditional second moments')
ax.set_ylabel('252-day rolling volatility of Mkt-RF, % annualized')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f'{FIG}/ff_rolling_vol.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 3. Kernel regression: a non-parametric conditional mean

The Nadaraya-Watson estimator is a weighted average:

$$
\widehat m(x)=\frac{\sum_{t=1}^T K\left((X_t-x)/h\right)y_t}{\sum_{t=1}^T K\left((X_t-x)/h\right)}.
$$

The key choice is the bandwidth $h$: a small bandwidth lowers bias but raises variance; a large bandwidth smooths away nonlinearities.

In [ ]:
app = ff.loc['1990':].copy()
app['mkt_lag1'] = app['Mkt-RF'].shift(1)
app['abs_mkt_lag1'] = app['Mkt-RF'].shift(1).abs()
app['mkt_next'] = app['Mkt-RF']
app = app.dropna()

def nw_kernel(x_train, y_train, x_grid, h):
    # Gaussian Nadaraya-Watson kernel estimator, written explicitly for teaching.
    out = []
    for x0 in x_grid:
        z = (x_train - x0) / h
        w = np.exp(-0.5 * z**2)
        s = w.sum()
        out.append(np.nan if s == 0 else (w @ y_train) / s)
    return np.asarray(out)

x = app['mkt_lag1'].values
y = app['mkt_next'].values
lo, hi = np.quantile(x, [0.01, 0.99])
grid = np.linspace(lo, hi, 120)
h = 1.06 * np.std(x) * len(x) ** (-1/5)

mhat = nw_kernel(x, y, grid, h)
lin = sm.OLS(y, sm.add_constant(x)).fit()
print('Bandwidth in basis points:', round(10000*h, 2))

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(100*x[::5], 100*y[::5], s=4, alpha=0.15, label='Daily observations')
plt.plot(100*grid, 100*mhat, linewidth=2.5, label='Kernel conditional mean')
plt.plot(100*grid, 100*lin.predict(sm.add_constant(grid)), linestyle='--', linewidth=2, label='Linear AR(1)')
plt.axhline(0, linewidth=0.8)
plt.title('Conditional mean: almost linear at the center, unstable in the tails')
plt.xlabel('Lagged Mkt-RF return, %')
plt.ylabel('Next-day Mkt-RF return, %')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/ff_kernel_conditional_mean.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 4. Kernel regression for volatility

The conditional mean of returns is hard to predict. The conditional scale is easier: large absolute returns tend to be followed by larger absolute returns. This is a nonlinear conditional moment even before using a GARCH model.

In [ ]:
xv = app['abs_mkt_lag1'].values
yv = app['mkt_next'].abs().values
lo, hi = np.quantile(xv, [0.01, 0.99])
gridv = np.linspace(lo, hi, 100)
hv = 1.06 * np.std(xv) * len(xv) ** (-1/5)
vol_resp = nw_kernel(xv, yv, gridv, hv)
linv = sm.OLS(yv, sm.add_constant(xv)).fit()

plt.figure(figsize=(9, 5))
plt.scatter(100*xv[::5], 100*yv[::5], s=4, alpha=0.13, label='Daily observations')
plt.plot(100*gridv, 100*vol_resp, linewidth=2.5, label='Kernel volatility response')
plt.plot(100*gridv, 100*linv.predict(sm.add_constant(gridv)), linestyle='--', linewidth=2, label='Linear approximation')
plt.title('Conditional volatility is nonlinear: large moves beget large moves')
plt.xlabel('Absolute lagged Mkt-RF return, %')
plt.ylabel('Absolute next-day Mkt-RF return, %')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/ff_kernel_vol_response.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 5. Threshold autoregression

A threshold model lets the intercept and slope change when a transition variable crosses a threshold:

$$
y_t = \alpha_1 + \phi_1 y_{t-1} + u_t \quad \text{if } y_{t-1}\le \tau,
$$

$$
y_t = \alpha_2 + \phi_2 y_{t-1} + u_t \quad \text{if } y_{t-1}> \tau.
$$

We estimate the threshold by grid search. The point is not that this is the best forecast model; the point is that nonlinearity can be estimated with ordinary least squares conditional on a candidate threshold.

In [ ]:
Y = app['mkt_next'].values
Xlag = app['mkt_lag1'].values
qs = np.linspace(0.10, 0.90, 81)
thresholds = np.quantile(Xlag, qs)
rows = []

for tau in thresholds:
    I = (Xlag <= tau).astype(float)
    Xtar = np.column_stack([np.ones_like(Xlag), Xlag, I, Xlag*I])
    fit = sm.OLS(Y, Xtar).fit()
    ssr = np.sum(fit.resid**2)
    k = Xtar.shape[1]
    aic = len(Y)*np.log(ssr/len(Y)) + 2*k
    rows.append([tau, ssr, aic])

grid_df = pd.DataFrame(rows, columns=['threshold', 'SSR', 'AIC'])
best = grid_df.loc[grid_df['AIC'].idxmin()]
best

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(100*grid_df['threshold'], grid_df['AIC'])
plt.axvline(100*best['threshold'], linestyle='--', linewidth=1.2, label=f"Selected threshold: {100*best['threshold']:.2f}%")
plt.title('Threshold search for a SETAR-style market equation')
plt.xlabel('Candidate threshold for lagged Mkt-RF, %')
plt.ylabel('AIC, lower is better')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/ff_threshold_search.pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
I = (Xlag <= best['threshold']).astype(float)
Xtar = np.column_stack([np.ones_like(Xlag), Xlag, I, Xlag*I])
tar = sm.OLS(Y, Xtar).fit()

pred_tar = tar.predict(np.column_stack([
    np.ones_like(grid), grid, (grid <= best['threshold']).astype(float), grid*(grid <= best['threshold']).astype(float)
]))

bins = np.quantile(Xlag, np.linspace(0.01, 0.99, 25))
bin_ids = np.digitize(Xlag, bins)
bin_x, bin_y = [], []
for b in np.unique(bin_ids):
    m = bin_ids == b
    if m.sum() > 20:
        bin_x.append(Xlag[m].mean())
        bin_y.append(Y[m].mean())

plt.figure(figsize=(9, 5))
plt.scatter(100*np.asarray(bin_x), 100*np.asarray(bin_y), s=28, label='Binned conditional averages')
plt.plot(100*grid, 100*lin.predict(sm.add_constant(grid)), linestyle='--', linewidth=2, label='Linear AR(1)')
plt.plot(100*grid, 100*pred_tar, linewidth=2.5, label='Estimated TAR')
plt.axvline(100*best['threshold'], linestyle=':', linewidth=1.2)
plt.axhline(0, linewidth=0.8)
plt.title('Threshold models let slopes change across states')
plt.xlabel('Lagged Mkt-RF return, %')
plt.ylabel('Next-day Mkt-RF return, %')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/ff_tar_fit.pdf', bbox_inches='tight')
plt.show()
plt.close()

print('Linear AR(1) slope:', round(lin.params[1], 3))
print('TAR low-regime slope:', round(tar.params[1] + tar.params[3], 3))
print('TAR high-regime slope:', round(tar.params[1], 3))

## 6. Conditional betas: nonlinear factor exposures

A factor can have one beta in normal markets and another beta in bad markets. This matters for portfolio construction because the risk contribution of a factor can be larger precisely when the market is falling.

In [ ]:
rows = []
for fac in ['SMB', 'HML', 'RMW', 'CMA']:
    yy = app[fac]
    m = app['Mkt-RF']
    down = m < np.quantile(m, 0.2)
    up = m > np.quantile(m, 0.8)
    beta_all = sm.OLS(yy, sm.add_constant(m)).fit().params['Mkt-RF']
    beta_down = sm.OLS(yy[down], sm.add_constant(m[down])).fit().params['Mkt-RF']
    beta_up = sm.OLS(yy[up], sm.add_constant(m[up])).fit().params['Mkt-RF']
    rows.append([fac, beta_all, beta_down, beta_up])

betas = pd.DataFrame(rows, columns=['Factor', 'Full-sample beta', 'Downside beta', 'Upside beta']).set_index('Factor')
betas.round(3)

In [ ]:
ax = betas[['Downside beta', 'Upside beta']].plot(kind='bar', figsize=(9, 5), rot=0)
ax.axhline(0, linewidth=0.8)
ax.set_title('Conditional betas are state-dependent')
ax.set_ylabel('Beta to Mkt-RF')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f'{FIG}/ff_downside_betas.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 7. Quantile regression: modeling the whole conditional distribution

OLS estimates the conditional mean. Quantile regression estimates conditional quantiles by minimizing an asymmetric absolute loss:

$$
\widehat\beta(\tau)=\arg\min_\beta \sum_{t=1}^T \rho_\tau(y_t-x_t'\beta),
$$

where $\rho_\tau(u)=u(\tau-\mathbf{1}_{u<0})$. This is useful in finance because downside and upside tails are not mirror images.

In [ ]:
qapp = app.copy()
qapp['lag_abs'] = qapp['Mkt-RF'].shift(1).abs()
qapp['lag_mkt'] = qapp['Mkt-RF'].shift(1)
qapp = qapp.dropna()

# Keep the quantile-regression illustration fast and stable for classroom use.
qapp = qapp.loc['2010':].copy()
Yq = qapp['Mkt-RF']
Xq = sm.add_constant(qapp[['lag_mkt', 'lag_abs']])
quantiles = np.arange(0.10, 0.91, 0.10)
coefs, lows, highs = [], [], []
for q in quantiles:
    mod = QuantReg(Yq, Xq).fit(q=q, max_iter=500)
    ci = mod.conf_int().loc['lag_abs']
    coefs.append(mod.params['lag_abs'])
    lows.append(ci[0])
    highs.append(ci[1])

plt.figure(figsize=(9, 5))
plt.plot(quantiles, coefs, marker='o', label='Coefficient on lagged |Mkt-RF|')
plt.fill_between(quantiles, lows, highs, alpha=0.2, label='95% interval')
plt.axhline(0, linewidth=0.8)
plt.title('Quantile regression: nonlinear effects across the distribution')
plt.xlabel('Conditional quantile of Mkt-RF')
plt.ylabel('Slope coefficient')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/ff_quantile_regression.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 8. Smooth transition model: from regimes to continuous nonlinearity

A logistic smooth transition autoregression replaces the hard threshold indicator by a smooth transition function:

$$
G(z_t;\gamma,c)=\left(1+\exp[-\gamma(z_t-c)]\right)^{-1}.
$$

When $\gamma$ is small, the model is close to linear. When $\gamma$ is large, the model becomes close to a threshold model.

In [ ]:
z = np.linspace(-3, 3, 200)
plt.figure(figsize=(9, 5))
for gamma in [1, 3, 10]:
    G = 1 / (1 + np.exp(-gamma*(z-0)))
    plt.plot(z, G, label=f'gamma={gamma}')
plt.title('Logistic smooth transition: from linear to threshold-like')
plt.xlabel('Transition variable')
plt.ylabel('Transition weight G(z)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/lstar_transition.pdf', bbox_inches='tight')
plt.show()
plt.close()

## 9. Takeaways

1. Nonlinearity in finance is usually more visible in risk than in expected returns.
2. Parametric nonlinear models impose a structure that is interpretable and testable.
3. Non-parametric estimators are excellent diagnostic tools, but they suffer from bandwidth choice and the curse of dimensionality.
4. The right workflow is: linear benchmark, diagnostic test, nonlinear specification, out-of-sample validation, economic interpretation.